# N13 — Discussion Robustness Checks

Four targeted robustness checks whose numerical outputs anchor specific claims
in the Discussion chapter.

| # | Analysis | Discussion claim |
|---|---|---|
| 1 | Constrained M1/M2/M3 within FF49 | How much of each model's gain comes from crossing industry boundaries? |
| 2 | M0 at k=10 (random + top-by-mktcap) | Is M1's advantage from peer-list-size reduction alone? |
| 3 | α* across k on validation years | Does the optimal fusion weight shift with peer-list size? |
| 4 | Exclude financials from candidate pool | Does retaining financials distort peer selection? |

**Metric:** MdAPE on $\ln(\text{EV/Sales})$ at $k = K\_\mathrm{MAIN}$ unless varied.


In [1]:
# Cell 1 — Imports & config (robust auto-install)
import subprocess, sys, importlib, warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')

def ensure(pkg, name=None):
    mod = name or pkg
    try:
        return importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])
        importlib.invalidate_caches()
        return importlib.import_module(mod)

pd = ensure('pandas'); np = ensure('numpy'); plt = ensure('matplotlib', 'matplotlib.pyplot')
ensure('pyarrow')

try:
    from scipy import stats as sp_stats
except (ImportError, ValueError):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           '--force-reinstall', '--no-deps', 'scipy'])
    importlib.invalidate_caches()
    for m in list(sys.modules):
        if m.startswith('scipy'): del sys.modules[m]
    from scipy import stats as sp_stats

notebook_dir = Path('__file__').parent if '__file__' in dir() else Path.cwd()
repo_root = next((p for p in [notebook_dir, *notebook_dir.parents]
                  if (p / 'config.py').exists()), None)
sys.path.insert(0, str(repo_root))
from config import *

pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
print(f"Config loaded. Repo: {repo_root}")


Config loaded. Repo: /work/Repo


## Cell 2 — Load data and compute baseline per-firm APE

Recomputes per-firm APE from the peer parquet files and sanity-checks against
the N10 chapter-level values. Also loads the selected-features list with a
defensive coercion so we work with a flat list of column names regardless of
whether the JSON file stores a list, a dict with `{"selected_features": [...]}`,
or a dict keyed by feature name.


In [2]:
peers_m0 = pd.read_parquet(PEERS_M0)
peers_m1 = pd.read_parquet(PEERS_M1)
peers_m2 = pd.read_parquet(PEERS_M2)
peers_m3 = pd.read_parquet(PEERS_M3)
multiples = pd.read_parquet(MULTIPLES)
panel = pd.read_parquet(PANEL_CLEAN,
    columns=['tic', 'fyear', 'industry', 'ff49_num', 'sic', 'market_cap', 'sale', 'at'])
finn = pd.read_parquet(FINANCIALS_NORM)
embeddings = pd.read_parquet(EMBEDDINGS)

# ── Load selected features defensively ────────────────────────────────────
# JSON could be: a flat list, a dict with 'selected_features' key, or
# a dict keyed by feature name. Coerce to a flat list of column names.
with open(SELECTED_FEATURES_FILE) as f:
    raw = json.load(f)

if isinstance(raw, list):
    selected_features = raw
elif isinstance(raw, dict):
    if 'selected_features' in raw and isinstance(raw['selected_features'], list):
        selected_features = raw['selected_features']
    elif all(isinstance(v, (int, float, bool, str)) for v in raw.values()):
        # Dict keyed by feature name with scalar values — take the keys
        selected_features = list(raw.keys())
    else:
        raise ValueError(f"selected_features JSON structure not recognised: "
                         f"top-level keys = {list(raw.keys())[:5]}...")
else:
    raise TypeError(f"selected_features: unexpected type {type(raw).__name__}")

# Verify the features actually exist in finn
missing_feats = [f for f in selected_features if f not in finn.columns]
assert not missing_feats, f"{len(missing_feats)} features missing from finn: {missing_feats[:5]}..."
print(f"Selected features: {len(selected_features)} columns loaded and verified in finn.")

# Restrict kNN models to top-K_MAIN for baseline
peers_m1 = peers_m1[peers_m1['rank'] <= K_MAIN].copy()
peers_m2 = peers_m2[peers_m2['rank'] <= K_MAIN].copy()
peers_m3 = peers_m3[peers_m3['rank'] <= K_MAIN].copy()

# Per-firm APE on ln(EV/Sales)
peer_ln = multiples[['tic', 'fyear', 'ln_v2s']].rename(
    columns={'tic': 'peer_tic', 'fyear': 'focal_fyear', 'ln_v2s': 'peer_ln_v2s'})
focal_ln = multiples[['tic', 'fyear', 'ln_v2s']].rename(
    columns={'tic': 'focal_tic', 'fyear': 'focal_fyear', 'ln_v2s': 'actual_ln'})

def compute_ape_from_peers(peers_df, label):
    merged = peers_df.merge(peer_ln, on=['peer_tic', 'focal_fyear'], how='left')
    pred = (merged.groupby(['focal_tic', 'focal_fyear'])['peer_ln_v2s']
            .median().reset_index()
            .rename(columns={'peer_ln_v2s': f'pred_{label}'}))
    pred = pred.merge(focal_ln, on=['focal_tic', 'focal_fyear'], how='left')
    pred = pred.dropna(subset=['actual_ln', f'pred_{label}'])
    pred = pred[pred['actual_ln'] != 0]
    pred[f'ape_{label}'] = (pred['actual_ln'] - pred[f'pred_{label}']).abs() / pred['actual_ln'].abs()
    return pred[['focal_tic', 'focal_fyear', f'ape_{label}']]

ape_all = compute_ape_from_peers(peers_m0, 'm0')
for name, df in [('m1', peers_m1), ('m2', peers_m2), ('m3', peers_m3)]:
    ape_all = ape_all.merge(compute_ape_from_peers(df, name),
                            on=['focal_tic', 'focal_fyear'], how='outer')

print(f"\nAPE frame: {len(ape_all):,} firm-years")
print(f"Sanity check (MdAPE %, should match N10):")
for m in ['m0', 'm1', 'm2', 'm3']:
    print(f"  {m.upper()}: {ape_all[f'ape_{m}'].median()*100:.2f}%")

# Attach focal attributes
panel_focal = panel.rename(columns={'tic':'focal_tic', 'fyear':'focal_fyear'})
ape_all = ape_all.merge(
    panel_focal[['focal_tic', 'focal_fyear', 'ff49_num', 'industry', 'market_cap']],
    on=['focal_tic', 'focal_fyear'], how='left')

# Canonical eval-sample set
eval_index = set(zip(ape_all['focal_tic'], ape_all['focal_fyear']))
all_results = []
print(f"\nEval sample: {len(eval_index):,} (focal_tic, focal_fyear) pairs")


Selected features: 64 columns loaded and verified in finn.

APE frame: 13,559 firm-years
Sanity check (MdAPE %, should match N10):
  M0: 54.79%
  M1: 43.75%
  M2: 51.89%
  M3: 41.13%

Eval sample: 13,559 (focal_tic, focal_fyear) pairs


## Section 1 — Constrained peer selection (within-industry)

Rebuilds M1, M2, M3 top-k peer lists where the candidate pool is restricted to
the focal firm's own FF49 industry. Focals whose industry has fewer than
$K_\mathrm{MAIN}$ other firms in that fyear are dropped (attrition reported).

For efficient computation, we use vectorised cosine similarity per
(fyear, industry) group. The original loop-based implementation was $O(n^2)$
per group and iterated inside a Python loop over focals, which is prohibitively
slow on 13,559 firm-years.


In [3]:
# Eligible focals — industries with >= K_MAIN firms in each fyear
eval_df = ape_all[['focal_tic', 'focal_fyear', 'ff49_num']].drop_duplicates()

ind_counts = (eval_df.groupby(['ff49_num', 'focal_fyear'])
              .size().rename('ind_n').reset_index())
eval_with_ind = eval_df.merge(ind_counts, on=['ff49_num', 'focal_fyear'], how='left')
eligible_mask = eval_with_ind['ind_n'] >= K_MAIN + 1  # +1 because focal is excluded from its own peer pool
eligible_focals = set(zip(
    eval_with_ind.loc[eligible_mask, 'focal_tic'],
    eval_with_ind.loc[eligible_mask, 'focal_fyear']))

print(f"Eval sample: {len(eval_index):,} focals")
print(f"Eligible for within-industry analysis (ind n >= {K_MAIN+1}): {len(eligible_focals):,}")
print(f"Dropped (thin industry): {len(eval_index) - len(eligible_focals):,}")


Eval sample: 13,559 focals
Eligible for within-industry analysis (ind n >= 11): 13,090
Dropped (thin industry): 469


In [4]:
def build_constrained_peers_vectorised(feature_df, feature_cols, k, eligible_focals):
    '''Vectorised within-industry top-k peer list from a feature matrix.
    feature_df: focal_tic, focal_fyear, ff49_num, + feature_cols.
    Only produces output for focals in `eligible_focals`.
    '''
    out_chunks = []
    eligible_by_year = {}
    for ft, fy in eligible_focals:
        eligible_by_year.setdefault(fy, set()).add(ft)

    for (fyear, ff49), grp in feature_df.groupby(['focal_fyear', 'ff49_num']):
        if len(grp) < k + 1:
            continue
        tickers = grp['focal_tic'].to_numpy()
        X = grp[feature_cols].to_numpy(dtype=float)

        # L2-normalise and compute full similarity matrix in one shot
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        Xn = X / np.where(norms > 0, norms, 1.0)
        sim = Xn @ Xn.T                    # (n, n)
        np.fill_diagonal(sim, -np.inf)

        # Take top-k indices per row via argpartition (O(n) per row)
        top_idx = np.argpartition(-sim, k, axis=1)[:, :k]
        row_idx = np.arange(len(tickers))[:, None]
        top_sims = sim[row_idx, top_idx]

        # Sort those k entries per row by similarity descending
        order = np.argsort(-top_sims, axis=1)
        top_idx_sorted = np.take_along_axis(top_idx, order, axis=1)
        top_sim_sorted = np.take_along_axis(top_sims, order, axis=1)

        # Filter to eligible focals only (avoids building rows we will drop)
        eligible_this_year = eligible_by_year.get(int(fyear), set())
        keep_rows = np.array([t in eligible_this_year for t in tickers])
        if not keep_rows.any():
            continue

        focals = tickers[keep_rows]
        peers  = tickers[top_idx_sorted[keep_rows]]
        sims_k = top_sim_sorted[keep_rows]

        n_focals = len(focals)
        out_chunks.append(pd.DataFrame({
            'focal_tic':   np.repeat(focals, k),
            'focal_fyear': np.int64(fyear),
            'peer_tic':    peers.reshape(-1),
            'rank':        np.tile(np.arange(1, k+1), n_focals),
            'similarity_score': sims_k.reshape(-1),
        }))
    return pd.concat(out_chunks, ignore_index=True) if out_chunks else pd.DataFrame(
        columns=['focal_tic','focal_fyear','peer_tic','rank','similarity_score'])

# Build feature frame for M1 with FF49 attached (load FF49 from panel directly)
finn_eval = finn.rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'}).copy()
finn_eval = finn_eval.drop(columns=[c for c in finn_eval.columns
                                    if c.lower() in ('ff49_num', 'ff49')], errors='ignore')
ff49_map = (panel.rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'})
                  [['focal_tic', 'focal_fyear', 'ff49_num']])
finn_eval = finn_eval.merge(ff49_map, on=['focal_tic', 'focal_fyear'], how='left')
assert 'ff49_num' in finn_eval.columns, f"merge failed; columns: {finn_eval.columns.tolist()[:10]}..."
finn_eval = finn_eval.dropna(subset=['ff49_num'])
finn_eval = finn_eval[finn_eval.apply(lambda r: (r['focal_tic'], r['focal_fyear']) in eval_index, axis=1)]

print(f"finn_eval frame: {len(finn_eval):,} rows, "
      f"{finn_eval['focal_tic'].nunique():,} focals, "
      f"{finn_eval['focal_fyear'].nunique()} fyears")

# Build M1 constrained
print("\nM1 constrained: building within-industry top-K_MAIN ...")
peers_m1_c = build_constrained_peers_vectorised(
    finn_eval, selected_features, K_MAIN, eligible_focals)
print(f"  rows: {len(peers_m1_c):,}  focals: {peers_m1_c['focal_tic'].nunique():,}")


finn_eval frame: 13,559 rows, 3,494 focals, 5 fyears

M1 constrained: building within-industry top-K_MAIN ...
  rows: 130,900  focals: 3,388


In [5]:
# M2 constrained — use embedding columns from EMBEDDINGS file
emb_cols = [c for c in embeddings.columns if c not in ('tic', 'fyear')]
# If embeddings are stored as a single vector column (common pattern) this would fail
# but our file has columns emb_0..emb_767 per N7 convention
assert len(emb_cols) > 10, f"expected many embedding cols; found {len(emb_cols)}"

emb_eval = embeddings.rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'}).copy()
emb_eval = emb_eval.merge(ff49_map, on=['focal_tic', 'focal_fyear'], how='left')
emb_eval = emb_eval.dropna(subset=['ff49_num'])
emb_eval = emb_eval[emb_eval.apply(lambda r: (r['focal_tic'], r['focal_fyear']) in eval_index, axis=1)]

print(f"emb_eval frame: {len(emb_eval):,} rows, {len(emb_cols)} embedding dims")
print("\nM2 constrained: building within-industry top-K_MAIN ...")
peers_m2_c = build_constrained_peers_vectorised(
    emb_eval, emb_cols, K_MAIN, eligible_focals)
print(f"  rows: {len(peers_m2_c):,}  focals: {peers_m2_c['focal_tic'].nunique():,}")

# M3 constrained — late fusion of constrained M1 + constrained M2
try:
    with open(ALPHA_OPTIMAL) as f:
        alpha_opt = json.load(f)
    alpha = alpha_opt.get('best_alpha', 0.3)
except Exception:
    alpha = 0.3
print(f"\nM3 constrained: using alpha = {alpha}")

def build_m3_from_pair(m1_df, m2_df, alpha, k_out=K_MAIN, k_cand=20):
    '''Weighted rank fusion from pre-built M1 and M2 peer lists.'''
    m1 = m1_df[m1_df['rank'] <= k_cand][['focal_tic','focal_fyear','peer_tic','rank']].rename(
        columns={'rank':'r_fin'})
    m2 = m2_df[m2_df['rank'] <= k_cand][['focal_tic','focal_fyear','peer_tic','rank']].rename(
        columns={'rank':'r_text'})
    fused = m1.merge(m2, on=['focal_tic','focal_fyear','peer_tic'], how='outer')
    fused['score'] = (
        alpha * np.where(fused['r_text'].notna(), 1.0/fused['r_text'], 0.0) +
        (1 - alpha) * np.where(fused['r_fin'].notna(), 1.0/fused['r_fin'], 0.0)
    )
    fused = fused.sort_values(['focal_tic','focal_fyear','score'], ascending=[True,True,False])
    fused['rank'] = fused.groupby(['focal_tic','focal_fyear']).cumcount() + 1
    fused = fused[fused['rank'] <= k_out].copy()
    fused['similarity_score'] = fused['score']
    return fused[['focal_tic','focal_fyear','peer_tic','rank','similarity_score']]

# Need k=20 candidates within industry for the fusion to have candidates to reorder
print("\nM3 constrained: building k=20 within-industry candidates per modality ...")
peers_m1_c_k20 = build_constrained_peers_vectorised(
    finn_eval, selected_features, 20, eligible_focals)
peers_m2_c_k20 = build_constrained_peers_vectorised(
    emb_eval, emb_cols, 20, eligible_focals)
peers_m3_c = build_m3_from_pair(peers_m1_c_k20, peers_m2_c_k20, alpha)
print(f"  rows: {len(peers_m3_c):,}  focals: {peers_m3_c['focal_tic'].nunique():,}")


emb_eval frame: 13,559 rows, 768 embedding dims

M2 constrained: building within-industry top-K_MAIN ...
  rows: 130,900  focals: 3,388

M3 constrained: using alpha = 0.3

M3 constrained: building k=20 within-industry candidates per modality ...
  rows: 128,230  focals: 3,315


In [6]:
# Shared eval function, restricted to eligible focals for apples-to-apples comparison
def eval_ape(peers_df, label, restrict_focals=None):
    merged = peers_df.merge(peer_ln, on=['peer_tic', 'focal_fyear'], how='left')
    pred = (merged.groupby(['focal_tic', 'focal_fyear'])['peer_ln_v2s']
            .median().reset_index()
            .rename(columns={'peer_ln_v2s': f'pred_{label}'}))
    pred = pred.merge(focal_ln, on=['focal_tic', 'focal_fyear'], how='left')
    if restrict_focals is not None:
        key = list(zip(pred['focal_tic'], pred['focal_fyear']))
        pred = pred[[k in restrict_focals for k in key]]
    pred = pred.dropna(subset=['actual_ln', f'pred_{label}'])
    pred = pred[pred['actual_ln'] != 0]
    pred[f'ape_{label}'] = (pred['actual_ln'] - pred[f'pred_{label}']).abs() / pred['actual_ln'].abs()
    return pred

r0  = eval_ape(peers_m0,   'm0',   restrict_focals=eligible_focals)
r1  = eval_ape(peers_m1,   'm1',   restrict_focals=eligible_focals)
r2  = eval_ape(peers_m2,   'm2',   restrict_focals=eligible_focals)
r3  = eval_ape(peers_m3,   'm3',   restrict_focals=eligible_focals)
r1c = eval_ape(peers_m1_c, 'm1c',  restrict_focals=eligible_focals)
r2c = eval_ape(peers_m2_c, 'm2c',  restrict_focals=eligible_focals)
r3c = eval_ape(peers_m3_c, 'm3c',  restrict_focals=eligible_focals)

print("== SECTION 1 — Within-industry MdAPE on ln(EV/Sales), eligible focals only ==")
print(f"{'Model':<34}{'MdAPE (%)':>12}{'n':>10}")
print("-" * 56)
for name, df, col in [
    ('M0 FF49 baseline (unchanged)',    r0,  'ape_m0'),
    ('M1 original (cross-industry)',    r1,  'ape_m1'),
    ('M1 constrained (within-ind.)',    r1c, 'ape_m1c'),
    ('M2 original (cross-industry)',    r2,  'ape_m2'),
    ('M2 constrained (within-ind.)',    r2c, 'ape_m2c'),
    ('M3 original (cross-industry)',    r3,  'ape_m3'),
    ('M3 constrained (within-ind.)',    r3c, 'ape_m3c'),
]:
    md_ = df[col].median() * 100 if len(df) else np.nan
    print(f"{name:<34}{md_:>11.2f}%{len(df):>10,}")
    all_results.append({'section': 1, 'variant': name, 'mdape_pct': md_, 'n': len(df)})

m0      = r0 ['ape_m0' ].median() * 100
m1_orig = r1 ['ape_m1' ].median() * 100
m1_con  = r1c['ape_m1c'].median() * 100
m3_orig = r3 ['ape_m3' ].median() * 100
m3_con  = r3c['ape_m3c'].median() * 100

print(f"\nVerdict (does M1's gain come from crossing industry boundaries?):")
print(f"  M0 MdAPE: {m0:.2f}%")
print(f"  M1 original gain over M0: {m0 - m1_orig:+.2f}pp")
print(f"  M1 constrained gain over M0: {m0 - m1_con:+.2f}pp")
share = (m0 - m1_con) / (m0 - m1_orig) * 100 if (m0 - m1_orig) != 0 else np.nan
print(f"  Share of M1's gain retained when constrained within industry: {share:.1f}%")


== SECTION 1 — Within-industry MdAPE on ln(EV/Sales), eligible focals only ==
Model                                MdAPE (%)         n
--------------------------------------------------------
M0 FF49 baseline (unchanged)            53.86%    13,090
M1 original (cross-industry)            43.10%    13,090
M1 constrained (within-ind.)            46.16%    13,090
M2 original (cross-industry)            51.26%    13,090
M2 constrained (within-ind.)            51.01%    13,090
M3 original (cross-industry)            40.73%    13,090
M3 constrained (within-ind.)            44.71%    12,823

Verdict (does M1's gain come from crossing industry boundaries?):
  M0 MdAPE: 53.86%
  M1 original gain over M0: +10.77pp
  M1 constrained gain over M0: +7.70pp
  Share of M1's gain retained when constrained within industry: 71.6%


## Section 2 — M0 at k=10 (two variants)

Tests whether M1's gain over M0 is a mechanical consequence of using a
smaller, more focused peer list.

- **Variant A:** Random k=10 within-industry M0, averaged over 50 seeds.
- **Variant B:** Top-10 by market cap within industry.


In [7]:
# Build industry → focal-ticker directory ONCE (the old version rebuilt it every
# iteration, which made 50 seeds prohibitively slow on 13,559 firms).
eligible_panel = panel_focal[
    panel_focal.apply(lambda r: (r['focal_tic'], r['focal_fyear']) in eligible_focals, axis=1)]
industry_members = (eligible_panel
    .groupby(['ff49_num', 'focal_fyear'])['focal_tic']
    .apply(list).to_dict())

# Precompute (focal → (ff49, fyear, candidates-list-minus-self))
focal_to_candidates = {}
for ft, fy in eligible_focals:
    row = eligible_panel[(eligible_panel['focal_tic']==ft) & (eligible_panel['focal_fyear']==fy)]
    if row.empty: continue
    ff49 = row['ff49_num'].iloc[0]
    cands = industry_members.get((ff49, fy), [])
    cands = [t for t in cands if t != ft]
    if cands:
        focal_to_candidates[(ft, fy)] = (cands, row['market_cap'].iloc[0])

print(f"Precomputed candidate pools for {len(focal_to_candidates):,} eligible focals")

# Variant A: random k=10 averaged over 50 seeds
BOOTSTRAP_DRAWS = 50

def random_draw_peers(seed):
    rng = np.random.default_rng(seed)
    rows = []
    for (ft, fy), (cands, _) in focal_to_candidates.items():
        k = min(K_MAIN, len(cands))
        picked = rng.choice(cands, size=k, replace=False)
        for r, peer in enumerate(picked, start=1):
            rows.append((ft, fy, peer, r, 1.0))
    return pd.DataFrame(rows, columns=['focal_tic','focal_fyear','peer_tic','rank','similarity_score'])

print(f"\nVariant A — random k=10 within-industry, averaged over {BOOTSTRAP_DRAWS} seeds ...")
mdapes_random = []
for b in range(BOOTSTRAP_DRAWS):
    pdf = random_draw_peers(seed=b)
    r = eval_ape(pdf, 'm0_rand', restrict_focals=eligible_focals)
    mdapes_random.append(r['ape_m0_rand'].median() * 100)

mdape_rand_mean = float(np.mean(mdapes_random))
mdape_rand_std  = float(np.std(mdapes_random))
print(f"  MdAPE (mean over {BOOTSTRAP_DRAWS} seeds): {mdape_rand_mean:.2f}% (sd={mdape_rand_std:.3f}pp)")
print(f"  seed range: [{min(mdapes_random):.2f}, {max(mdapes_random):.2f}]")


Precomputed candidate pools for 13,090 eligible focals

Variant A — random k=10 within-industry, averaged over 50 seeds ...
  MdAPE (mean over 50 seeds): 57.83% (sd=0.447pp)
  seed range: [56.89, 58.96]


In [12]:
# Variant B: closest 10 by market cap within industry
# For each focal, pick the k industry peers whose market cap is closest to
# the focal's own market cap (measured on log scale for robustness to scale).
mkt_lookup = panel_focal.set_index(['focal_tic','focal_fyear'])['market_cap'].to_dict()

def closest_by_mktcap_peers():
    rows = []
    for (ft, fy), (cands, focal_mcap) in focal_to_candidates.items():
        if pd.isna(focal_mcap) or focal_mcap <= 0:
            continue
        log_focal = np.log(focal_mcap)
        # Distance in log-market-cap space
        cand_dists = []
        for t in cands:
            mcap = mkt_lookup.get((t, fy), np.nan)
            if pd.isna(mcap) or mcap <= 0:
                continue
            cand_dists.append((t, abs(np.log(mcap) - log_focal)))
        cand_dists.sort(key=lambda x: x[1])  # ascending distance
        picked = [t for t, _ in cand_dists[:K_MAIN]]
        for rnk, peer in enumerate(picked, start=1):
            rows.append((ft, fy, peer, rnk, 1.0))
    return pd.DataFrame(rows, columns=['focal_tic','focal_fyear','peer_tic','rank','similarity_score'])

print("Variant B — closest 10 by market cap within industry ...")
pdf_mcap = closest_by_mktcap_peers()
r_mcap = eval_ape(pdf_mcap, 'm0_mcap', restrict_focals=eligible_focals)
mdape_mcap = r_mcap['ape_m0_mcap'].median() * 100
print(f"  MdAPE: {mdape_mcap:.2f}%  n = {len(r_mcap):,}")

print(f"\n== SECTION 2 — M0 at k=10 variants vs M0 full-industry and M1 ==")
print(f"  M0 full-industry:                      {m0:.2f}%  (n = {len(r0):,})")
print(f"  M0 k=10 random (mean over B=50):       {mdape_rand_mean:.2f}%  (sd={mdape_rand_std:.2f}pp)")
print(f"  M0 k=10 closest-by-market-cap:         {mdape_mcap:.2f}%")
print(f"  M1 original:                           {m1_orig:.2f}%")

all_results.append({'section': 2, 'variant': 'M0 full-industry',
                    'mdape_pct': m0, 'n': len(r0)})
all_results.append({'section': 2, 'variant': 'M0 k=10 random (B=50 mean)',
                    'mdape_pct': mdape_rand_mean, 'n': len(eligible_focals)})
all_results.append({'section': 2, 'variant': 'M0 k=10 closest-by-market-cap',
                    'mdape_pct': mdape_mcap, 'n': len(r_mcap)})

shrink_gain = m0 - mdape_rand_mean
simi_gain_size = m0 - mdape_mcap           # gain from size-similarity over M0
simi_gain_fin  = mdape_mcap - m1_orig      # gain from financial similarity over size
total_gain = m0 - m1_orig

print(f"\nVerdict:")
print(f"  Gain from list-size reduction alone (M0 full -> k=10 random): {shrink_gain:+.2f}pp")
print(f"  Gain from size-similarity (M0 full -> closest-by-mktcap):     {simi_gain_size:+.2f}pp")
print(f"  Gain from multivariate financial similarity (size -> M1):     {simi_gain_fin:+.2f}pp")
if abs(total_gain) > 0.1:
    print(f"  Share of M1's gain that is list-size reduction: "
          f"{shrink_gain/total_gain*100:.1f}%")
    print(f"  Share of M1's gain that is already in size-similarity: "
          f"{(m0 - mdape_mcap)/total_gain*100:.1f}%")

Variant B — closest 10 by market cap within industry ...
  MdAPE: 52.64%  n = 13,090

== SECTION 2 — M0 at k=10 variants vs M0 full-industry and M1 ==
  M0 full-industry:                      53.86%  (n = 13,090)
  M0 k=10 random (mean over B=50):       57.83%  (sd=0.45pp)
  M0 k=10 closest-by-market-cap:         52.64%
  M1 original:                           43.10%

Verdict:
  Gain from list-size reduction alone (M0 full -> k=10 random): -3.97pp
  Gain from size-similarity (M0 full -> closest-by-mktcap):     +1.22pp
  Gain from multivariate financial similarity (size -> M1):     +9.55pp
  Share of M1's gain that is list-size reduction: -36.9%
  Share of M1's gain that is already in size-similarity: 11.3%


## Section 3 — α* across k on validation years

For each $k \in \{5, 10, 15, 20\}$, grid-search $\alpha$ on validation years 2020–2022.
Tests whether the optimal fusion weight drifts with peer-group size.


In [9]:
# Load full top-20 peer candidates for the fusion grid
peers_m1_k20 = pd.read_parquet(PEERS_M1)
peers_m2_k20 = pd.read_parquet(PEERS_M2)
peers_m1_k20 = peers_m1_k20[peers_m1_k20['rank'] <= 20].copy()
peers_m2_k20 = peers_m2_k20[peers_m2_k20['rank'] <= 20].copy()

val_mask = ape_all['focal_fyear'].isin(VALIDATION_YEARS)
val_focals = set(zip(ape_all.loc[val_mask, 'focal_tic'], ape_all.loc[val_mask, 'focal_fyear']))
print(f"Validation focals: {len(val_focals):,}")

alpha_grid = ALPHA_GRID
k_grid = [5, 10, 15, 20]
records = []

for k_test in k_grid:
    for a in alpha_grid:
        peers_m3_tmp = build_m3_from_pair(peers_m1_k20, peers_m2_k20, a,
                                          k_out=k_test, k_cand=20)
        r_tmp = eval_ape(peers_m3_tmp, 'm3_tmp', restrict_focals=val_focals)
        md_ = r_tmp['ape_m3_tmp'].median() * 100 if len(r_tmp) else np.nan
        records.append({'k': k_test, 'alpha': a,
                        'val_mdape_pct': md_, 'n': len(r_tmp)})

alpha_by_k = pd.DataFrame(records)
print("\nValidation MdAPE by (k, alpha):")
pivot = alpha_by_k.pivot_table(index='alpha', columns='k', values='val_mdape_pct')
print(pivot.round(2).to_string())

print(f"\nOptimal alpha per k:")
print(f"{'k':>4}{'alpha*':>10}{'val MdAPE (%)':>18}")
optimal_alpha_values = []
for k_test in k_grid:
    sub = alpha_by_k[alpha_by_k['k'] == k_test]
    best = sub.loc[sub['val_mdape_pct'].idxmin()]
    optimal_alpha_values.append(float(best['alpha']))
    print(f"{k_test:>4}{best['alpha']:>10.2f}{best['val_mdape_pct']:>17.2f}%")
    all_results.append({'section': 3,
                        'variant': f'k={k_test} alpha*={best["alpha"]:.1f}',
                        'mdape_pct': best['val_mdape_pct'], 'n': int(best['n'])})

if len(set(optimal_alpha_values)) == 1:
    print(f"\nVerdict: alpha* stable at {optimal_alpha_values[0]} across all k — no drift.")
else:
    drift = max(optimal_alpha_values) - min(optimal_alpha_values)
    print(f"\nVerdict: alpha* ranges from {min(optimal_alpha_values)} to {max(optimal_alpha_values)} (drift {drift:.1f}).")


Validation focals: 7,944

Validation MdAPE by (k, alpha):
k         5      10     15     20
alpha                            
0.0    44.45  43.23  43.00  42.34
0.1    44.28  42.24  42.51  41.65
0.2    43.17  41.13  41.03  40.83
0.3    43.16  40.67  41.05  40.65
0.4    43.01  41.17  40.88  40.85
0.5    43.27  41.62  41.54  41.17
0.6    44.80  42.29  42.53  42.62
0.7    48.39  44.05  44.51  44.33
0.8    49.14  45.92  46.30  46.20
0.9    52.35  48.91  49.49  47.69
1.0    52.45  51.07  50.91  50.22

Optimal alpha per k:
   k    alpha*     val MdAPE (%)
   5      0.40            43.01%
  10      0.30            40.67%
  15      0.40            40.88%
  20      0.30            40.65%

Verdict: alpha* ranges from 0.3 to 0.4 (drift 0.1).


## Section 4 — Financial / utilities exclusion

Drops FF49 industries 44 (Banking), 45 (Insurance), 47 (Trading), 48 (Other
Financial) and 31 (Utilities) from the candidate peer pool for M1 and M3.
Evaluation sample stays all eval firms — we test whether including these
industries as peers distorts valuations for non-financial focal firms.


In [10]:
EXCLUDE_FF49 = {44, 45, 47, 48, 31}

def drop_excluded_peers(peers_df, exclude_set):
    lookup = panel_focal[['focal_tic','focal_fyear','ff49_num']].rename(
        columns={'focal_tic':'peer_tic','ff49_num':'peer_ff49'})
    m = peers_df.merge(lookup, on=['peer_tic','focal_fyear'], how='left')
    m = m[~m['peer_ff49'].isin(exclude_set)]
    m = m.sort_values(['focal_tic','focal_fyear','rank'])
    m['rank'] = m.groupby(['focal_tic','focal_fyear']).cumcount() + 1
    return m[['focal_tic','focal_fyear','peer_tic','rank','similarity_score']]

# Rebuild with top-K_MAIN after exclusion; M3 needs top-20 pool for proper fusion
peers_m1_excl = drop_excluded_peers(peers_m1_k20, EXCLUDE_FF49)
peers_m1_excl = peers_m1_excl[peers_m1_excl['rank'] <= K_MAIN].copy()
peers_m2_excl_k20 = drop_excluded_peers(peers_m2_k20, EXCLUDE_FF49)
peers_m1_excl_k20 = drop_excluded_peers(peers_m1_k20, EXCLUDE_FF49)
peers_m3_excl = build_m3_from_pair(peers_m1_excl_k20, peers_m2_excl_k20, alpha)

r1_excl = eval_ape(peers_m1_excl, 'm1_excl')
r3_excl = eval_ape(peers_m3_excl, 'm3_excl')
r1_base = eval_ape(peers_m1, 'm1_base')
r3_base = eval_ape(peers_m3, 'm3_base')

md_m1_base = r1_base['ape_m1_base'].median() * 100
md_m1_excl = r1_excl['ape_m1_excl'].median() * 100
md_m3_base = r3_base['ape_m3_base'].median() * 100
md_m3_excl = r3_excl['ape_m3_excl'].median() * 100

print(f"== SECTION 4 — Financial / utilities exclusion from candidate pool ==")
print(f"{'':32}{'Baseline':>10}{'Excluded':>10}{'Delta':>10}")
print(f"  {'M1 MdAPE (all focals)':30}{md_m1_base:>9.2f}%{md_m1_excl:>9.2f}%{md_m1_excl-md_m1_base:>+9.2f}pp")
print(f"  {'M3 MdAPE (all focals)':30}{md_m3_base:>9.2f}%{md_m3_excl:>9.2f}%{md_m3_excl-md_m3_base:>+9.2f}pp")
print(f"  n (eval sample): {len(r1_base):,}")

# Non-financial focals only (cleaner comparison)
non_fin_focals = set(
    zip(panel_focal.loc[~panel_focal['ff49_num'].isin(EXCLUDE_FF49), 'focal_tic'],
        panel_focal.loc[~panel_focal['ff49_num'].isin(EXCLUDE_FF49), 'focal_fyear'])
) & eval_index

r1_base_nf = eval_ape(peers_m1, 'm1_base_nf', restrict_focals=non_fin_focals)
r1_excl_nf = eval_ape(peers_m1_excl, 'm1_excl_nf', restrict_focals=non_fin_focals)
r3_base_nf = eval_ape(peers_m3, 'm3_base_nf', restrict_focals=non_fin_focals)
r3_excl_nf = eval_ape(peers_m3_excl, 'm3_excl_nf', restrict_focals=non_fin_focals)

md_m1_base_nf = r1_base_nf['ape_m1_base_nf'].median()*100
md_m1_excl_nf = r1_excl_nf['ape_m1_excl_nf'].median()*100
md_m3_base_nf = r3_base_nf['ape_m3_base_nf'].median()*100
md_m3_excl_nf = r3_excl_nf['ape_m3_excl_nf'].median()*100

print(f"\nNon-financial focals only (n = {len(non_fin_focals):,}):")
print(f"  {'M1 MdAPE':30}{md_m1_base_nf:>9.2f}%{md_m1_excl_nf:>9.2f}%{md_m1_excl_nf-md_m1_base_nf:>+9.2f}pp")
print(f"  {'M3 MdAPE':30}{md_m3_base_nf:>9.2f}%{md_m3_excl_nf:>9.2f}%{md_m3_excl_nf-md_m3_base_nf:>+9.2f}pp")

for lbl, val in [
    ('M1 base (all focals)',      md_m1_base),
    ('M1 excl (all focals)',      md_m1_excl),
    ('M3 base (all focals)',      md_m3_base),
    ('M3 excl (all focals)',      md_m3_excl),
    ('M1 base (non-fin focals)',  md_m1_base_nf),
    ('M1 excl (non-fin focals)',  md_m1_excl_nf),
    ('M3 base (non-fin focals)',  md_m3_base_nf),
    ('M3 excl (non-fin focals)',  md_m3_excl_nf),
]:
    all_results.append({'section': 4, 'variant': lbl, 'mdape_pct': val, 'n': len(r1_base)})

print(f"\nVerdict: |Delta| < 1pp suggests retention is empirically defended.")


== SECTION 4 — Financial / utilities exclusion from candidate pool ==
                                  Baseline  Excluded     Delta
  M1 MdAPE (all focals)             43.75%    52.60%    +8.85pp
  M3 MdAPE (all focals)             41.13%    50.92%    +9.79pp
  n (eval sample): 13,559

Non-financial focals only (n = 9,961):
  M1 MdAPE                          50.11%    50.38%    +0.27pp
  M3 MdAPE                          47.34%    47.85%    +0.51pp

Verdict: |Delta| < 1pp suggests retention is empirically defended.


## Save results

In [11]:
results_df = pd.DataFrame(all_results)
out = DATA_RESULTS / 'n13_discussion_robustness.csv'
results_df.to_csv(out, index=False)
print(f"Saved: {out}")
print(f"\nFull results table:")
print(results_df.to_string(index=False))


Saved: /work/Repo/data/results/n13_discussion_robustness.csv

Full results table:
 section                      variant  mdape_pct     n
       1 M0 FF49 baseline (unchanged)  53.860720 13090
       1 M1 original (cross-industry)  43.095327 13090
       1 M1 constrained (within-ind.)  46.157655 13090
       1 M2 original (cross-industry)  51.255599 13090
       1 M2 constrained (within-ind.)  51.012680 13090
       1 M3 original (cross-industry)  40.730663 13090
       1 M3 constrained (within-ind.)  44.706518 12823
       2             M0 full-industry  53.860720 13090
       2   M0 k=10 random (B=50 mean)  57.827767 13090
       2       M0 k=10 top-market-cap  61.754929 13090
       3               k=5 alpha*=0.4  43.008250  7944
       3              k=10 alpha*=0.3  40.673565  7944
       3              k=15 alpha*=0.4  40.882472  7944
       3              k=20 alpha*=0.3  40.646990  7944
       4         M1 base (all focals)  43.749699 13559
       4         M1 excl (all focals) 

## Discussion-ready summary

**Section 1 — constrained peer selection.**
If M1's constrained-within-industry MdAPE is close to its unconstrained MdAPE,
the cross-industry matching does little work. If constrained M1 is close to M0,
cross-industry matching is essential to M1's advantage.

**Section 2 — M0 at k=10 variants.**
If both k=10 M0 variants land near M0 full-industry, then peer-list-size
reduction per se is not the source of M1's gain. If they substantially beat
M0 full-industry, then a share of M1's gain is attributable to list-size
reduction rather than similarity-criterion quality.

**Section 3 — α* across k.**
If α* is stable across k, the fusion weight is structural. If α* drifts
toward text at larger k, text is a diversifying signal; toward financial at
larger k, text is more useful at tight peer lists only.

**Section 4 — exclude financials.**
If Δ MdAPE is near zero, retention of financial industries is empirically
defended. If |Δ| > 1pp, retention materially affects results and should be
noted as a design limitation.
